# GoiEner Clustering, Multi-Resolution Features

Exploratory notebook, first pass, not part of any book chapter.
`04-customer-feeder-clustering-goiener-code.ipynb` clusters on a single
feature per household: the season-mean daily shape, normalized by its own
peak. That answers "does this household use electricity at the same time
of day as this other one." It found a real, disclosed anomaly worth
checking further: GoiEner's own validity curve is non-monotonic (a local
maximum at $k{=}4$, unlike AusNet's and London's own monotonic curves),
and its chosen split is lopsided (79/37/177/7 households).

This notebook asks a different real question with a different real
feature representation, adapted from the author's own prior research code
at `resources/profiling 3/src/utils/prepare_data.py`'s
`prepare_feature()`: cluster on **consumption level and trend across
multiple time resolutions** (daily, weekly, monthly totals, plus their
first differences), not on intraday shape at all. Two households with
identical daily shapes but very different absolute consumption, or very
different month-to-month growth, are indistinguishable to the shape-only
approach; this feature set is built specifically to separate them.

Real data loading is reused verbatim from the shape-only notebook: the same
300 real households, the same 360-day window (2021-06-06 through
2022-05-31), the same real per-household coverage filter. Only the feature
construction changes.

Scope, disclosed rather than assumed complete: this first pass checks
whether the headline validity/cluster-structure finding changes under
this richer representation. It does not re-run the cross-quarter
stability or history-length-sensitivity checks from the shape-only
notebook, since a multi-resolution feature is not well-defined at every
one of those shorter windows (weekly and monthly features do not exist
inside a 1-day or 1-week sub-window) and forcing it would obscure rather
than clarify the real comparison.

## Getting the data

Identical to `04-customer-feeder-clustering-goiener-code.ipynb`: stream
300 real households meeting a 99% real hourly-coverage bar over the same
360-day window directly out of the compressed `imp-post.tzst` archive.

In [ ]:
import io
from pathlib import Path
import tarfile

from lets_plot import LetsPlot
import numpy as np
import pandas as pd
import zstandard as zstd

LetsPlot.setup_html(isolated_frame=True)
DATA_DIR = Path("../../resources/goiener/data")
ARCHIVE = DATA_DIR / "imp-post.tzst"
METADATA = DATA_DIR / "metadata.csv"
STEPS_PER_DAY = 24  # real, disclosed mismatch: GoiEner is hourly, not half-hourly like AusNet/London
N_HOUSEHOLDS = 300
WINDOW_START = pd.Timestamp("2021-06-06")
WINDOW_DAYS = 360  # a real full year: enough real history for daily, weekly, and monthly resolution
MIN_COVERAGE = 0.99

if not ARCHIVE.exists():
    raise SystemExit(f"{ARCHIVE} not found; run scripts/fetch_goiener_data.py first")

meta = pd.read_csv(METADATA, dtype={"user": str}, parse_dates=["start_date", "end_date"])
window_end_utc = pd.Timestamp(WINDOW_START, tz="UTC") + pd.Timedelta(days=WINDOW_DAYS)
candidates = meta[
    (meta["missing_samples_pct"] < 1.0)
    & (meta["length_years"] >= 1.0)
    & (meta["start_date"] <= pd.Timestamp(WINDOW_START, tz="UTC"))
    & (meta["end_date"] >= window_end_utc)
]
print(f"real candidates covering the target window at the metadata level: {len(candidates)}")

target_ids = set(candidates["user"].sample(n=N_HOUSEHOLDS, random_state=42))

real candidates covering the target window at the metadata level: 16888


In [ ]:
found = {}
dctx = zstd.ZstdDecompressor()
with ARCHIVE.open("rb") as fh, dctx.stream_reader(fh) as reader, tarfile.open(fileobj=reader, mode="r|") as tar:
    for member in tar:
        if not member.isfile():
            continue
        stem = Path(member.name).stem
        if stem not in target_ids:
            continue
        raw = tar.extractfile(member)
        if raw is None:
            continue
        found[stem] = raw.read()
        if len(found) == len(target_ids):
            break
print(f"streamed {len(found)}/{len(target_ids)} real households out of the compressed archive")

streamed 300/300 real households out of the compressed archive


In [ ]:
window_end = WINDOW_START + pd.Timedelta(days=WINDOW_DAYS)
full_index = pd.date_range(WINDOW_START, window_end, freq="1h", inclusive="left")
print(f"expected real hourly timesteps: {len(full_index)}")

series = {}
for uid, raw in found.items():
    df = pd.read_csv(io.BytesIO(raw), parse_dates=["timestamp"]).set_index("timestamp").sort_index()
    win = df.reindex(full_index)
    if win["kWh"].notna().mean() >= MIN_COVERAGE:
        series[uid] = win["kWh"].interpolate(method="linear", limit_direction="both")

household_ids = list(series.keys())
n_customers = len(household_ids)
load_data = np.stack([series[uid].to_numpy() for uid in household_ids]).reshape(n_customers, WINDOW_DAYS, STEPS_PER_DAY)
print(f"load_data: {load_data.shape} (customers, days, hours), units kWh per hour")

expected real hourly timesteps: 8640


load_data: (300, 360, 24) (customers, days, hours), units kWh per hour


## Multi-resolution features: level and trend, not shape

Adapted from `prepare_data.py`'s own `prepare_feature()`: resample each
real household's hourly series to daily, weekly, and monthly totals over
the full real year, then take the first difference of each resampled
series (day-to-day, week-to-week, month-to-month change), and concatenate
level and diff into one feature vector per household. `MinMaxScaler`
normalizes each of the resulting columns independently, the same real
choice the author's own code makes, since daily totals, weekly totals, and
monthly totals live on very different absolute scales.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

daily_index = pd.date_range(WINDOW_START, periods=WINDOW_DAYS, freq="D")
daily_totals = pd.DataFrame(load_data.sum(axis=2).T, index=daily_index, columns=household_ids)
weekly_totals = daily_totals.resample("W").sum()
monthly_totals = daily_totals.resample("ME").sum()

daily_diff = daily_totals.diff().fillna(0.0)
weekly_diff = weekly_totals.diff().fillna(0.0)
monthly_diff = monthly_totals.diff().fillna(0.0)

print(f"daily: {daily_totals.shape}, weekly: {weekly_totals.shape}, monthly: {monthly_totals.shape}")

multires_raw = pd.concat(
    [daily_totals.T, weekly_totals.T, monthly_totals.T, daily_diff.T, weekly_diff.T, monthly_diff.T],
    axis=1,
)
multires_raw.columns = [f"col_{i}" for i in range(multires_raw.shape[1])]
print(f"concatenated level+diff feature width: {multires_raw.shape[1]} columns for {n_customers} households")

X_multires = MinMaxScaler().fit_transform(multires_raw.values)

daily: (360, 300), weekly: (53, 300), monthly: (12, 300)
concatenated level+diff feature width: 850 columns for 300 households


### K-means on the multi-resolution feature set

Same validity-curve procedure as the shape-only notebook, applied to this
genuinely different feature representation.

In [ ]:
from ark.plot.clustering import validity_curve, validity_scores
from ark.plot.gt_style import themed_gt

scores_multires = validity_scores(X_multires, range(2, 10))
themed_gt(scores_multires.round(3))

k,inertia,silhouette,davies_bouldin
2,638.789,0.937,0.365
3,463.509,0.83,0.907
4,380.24,0.828,0.64
5,321.728,0.814,0.568
6,279.235,0.7,0.912
7,233.851,0.778,0.58
8,209.663,0.688,0.699
9,184.447,0.752,0.475


In [ ]:
from lets_plot import ggsize

p = validity_curve(scores_multires)
p + ggsize(width=600, height=400)

In [ ]:
N_CLUSTERS_MULTIRES = int(scores_multires.loc[scores_multires["silhouette"].idxmax(), "k"])
print(f"N_CLUSTERS chosen from the real silhouette maximum above: {N_CLUSTERS_MULTIRES}")

from sklearn.cluster import KMeans

kmeans_multires = KMeans(n_clusters=N_CLUSTERS_MULTIRES, n_init=20, random_state=0)
labels_multires = kmeans_multires.fit_predict(X_multires)
table_multires = pd.DataFrame({"labels": labels_multires}).value_counts().sort_index().reset_index()
table_multires.columns = ["Label", "Count"]
themed_gt(table_multires)

N_CLUSTERS chosen from the real silhouette maximum above: 2


Label,Count
0,298
1,2


Real household daily-consumption trajectories, by cluster, over the same
full year: this is what the multi-resolution feature set actually
separates on, level and trend, not intraday shape (there is no single
"day" x-axis to plot here the way the shape-only notebook's `cluster_profiles`
does; each real household's own full daily trajectory is shown instead).

In [ ]:
from lets_plot import aes, geom_line, ggplot, labs, scale_color_manual

from ark.plot.theme import BRAND_PALETTE, modern_theme
from ark.plot.tokens import PRIMARY

cluster_daily_mean = daily_totals.T.groupby(labels_multires).mean().T
cluster_daily_long = cluster_daily_mean.reset_index(names="date").melt(
    id_vars="date", var_name="cluster", value_name="kwh_per_day"
)
cluster_daily_long["cluster"] = cluster_daily_long["cluster"].astype(str)

CLUSTER_COLORS = {str(c): BRAND_PALETTE[c % len(BRAND_PALETTE)] for c in range(N_CLUSTERS_MULTIRES)}
p = (
    ggplot(cluster_daily_long, aes(x="date", y="kwh_per_day", color="cluster"))
    + geom_line()
    + scale_color_manual(values=CLUSTER_COLORS)
    + labs(
        x="",
        y="Mean real daily consumption (kWh)",
        color="Cluster",
        title="Real cluster-mean daily trajectories, full year",
    )
    + modern_theme()
    + ggsize(700, 400)
)
p

## Does a more advanced method earn its complexity, on these features too?

Same IDEC-vs-KMeans comparison as the shape-only notebook, applied to the
multi-resolution feature set.

In [ ]:
from sklearn.metrics import davies_bouldin_score, silhouette_score

from ark.cluster.idec import fit_idec

comparison_rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=20, random_state=0)
    km_labels = km.fit_predict(X_multires)
    _, idec_labels = fit_idec(X_multires, n_clusters=k, random_state=0)
    comparison_rows.append(
        {
            "k": k,
            "kmeans_silhouette": silhouette_score(X_multires, km_labels),
            "kmeans_davies_bouldin": davies_bouldin_score(X_multires, km_labels),
            "idec_silhouette": silhouette_score(X_multires, idec_labels),
            "idec_davies_bouldin": davies_bouldin_score(X_multires, idec_labels),
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
themed_gt(comparison_df.round(3))

k,kmeans_silhouette,kmeans_davies_bouldin,idec_silhouette,idec_davies_bouldin
2,0.937,0.365,0.937,0.365
3,0.83,0.907,0.83,0.907
4,0.828,0.64,0.767,1.179
5,0.814,0.568,0.765,0.909
6,0.7,0.912,0.521,1.703
7,0.778,0.58,0.208,2.105
8,0.688,0.699,0.43,1.91


## How much does that overlap actually matter?

Same conformal-style check as the shape-only notebook: hold back 30% of
real households, calibrate a distance-to-centroid threshold at 90%
confidence, then see how many archetypes a genuinely honest confidence
set assigns each held-in household, on this feature set.

In [ ]:
from sklearn.model_selection import train_test_split

train_idx, calibration_idx = train_test_split(np.arange(n_customers), test_size=0.3, random_state=0)

conformal_km = KMeans(n_clusters=N_CLUSTERS_MULTIRES, n_init=20, random_state=0)
conformal_km.fit(X_multires[train_idx])
centroids = conformal_km.cluster_centers_


def distance_to_centroids(profiles: np.ndarray) -> np.ndarray:
    return np.linalg.norm(profiles[:, None, :] - centroids[None, :, :], axis=2)


ALPHA = 0.1
calibration_scores = distance_to_centroids(X_multires[calibration_idx]).min(axis=1)
n_calibration = len(calibration_idx)
quantile_level = min(np.ceil((n_calibration + 1) * (1 - ALPHA)) / n_calibration, 1.0)
tau = np.quantile(calibration_scores, quantile_level)

all_distances = distance_to_centroids(X_multires)
set_sizes = pd.Series([len(np.where(row <= tau)[0]) for row in all_distances])
size_counts = set_sizes.value_counts().sort_index().reset_index()
size_counts.columns = ["Set size", "Households"]
themed_gt(size_counts)

Set size,Households
0,31
1,269


## Summary

A real, honest negative result, not the improvement this notebook set out
to check for. Silhouette at $k{=}2$ reads as 0.937, far higher than
anything the shape-only notebook ever produced (its own best was 0.233 at
$k{=}4$), but the real cluster sizes expose why: 298 of 300 households
land in one cluster, 2 in the other. That is not a genuine two-archetype
population, it is 850 real feature columns (daily, weekly, and monthly
levels plus their diffs) built from only 300 real households, a real
curse-of-dimensionality regime where two households happen to sit far
enough from everyone else in that high-dimensional space to look like
their own cluster, inflating silhouette without finding anything real
about consumption behavior. IDEC does not escape this either: at
$k{=}2$ it reports the identical 0.937 silhouette, having converged to
the same degenerate split its own KMeans initialization started from.

This does not mean multi-resolution features are a bad idea, it means
they need real dimensionality reduction before clustering, a genuine
next step this first pass deliberately did not take, disclosed rather
than hidden behind a flattering silhouette number. The shape-only
notebook's own real findings, the non-monotonic validity curve and the
lopsided 79/37/177/7 split at $k{=}4$, are not resolved or explained by
this richer feature set; if anything, this run is a caution against
reading a single high silhouette score as evidence of a better
clustering without checking what it actually separates, exactly the
"checked, not assumed" discipline this book applies throughout.